# =======================================
# MOVIES DATASET EXPLORATORY DATA ANALYSIS
# =======================================

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
data = pd.read_csv('movies.csv')

### =============================================
### 2. DATA OVERVIEW
### =============================================

In [ ]:
data.head(10)

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
print(data.shape)

- We take look at missing data

In [ ]:
print("Missing values:\n------------------")
def missing_values():
    for col in data.columns:
        m_values = data[col].isna().sum()
        percentage = (m_values/data.shape[0])*100
        print(f'{col}: {percentage:.2f}% ({m_values})')
missing_values()

missing_percent = (data.isna().sum() / data.shape[0]) * 100
plt.figure(figsize=(8, 5))
plt.bar(missing_percent.index, missing_percent.values, color='darkgreen')
plt.xlabel("Stĺpce")
plt.ylabel("Percento chýbajúcich hodnôt")
plt.title("Chýbajúce hodnoty v datasete")
plt.xticks(rotation=45, ha="right")  
plt.tight_layout()

plt.show()

### =============================================
### 3. Data Cleaning
### =============================================

In [ ]:
data.head()

- First step will be removing all '\n' signs

In [ ]:
data['GENRE'] = data['GENRE'].str.replace('\n','').str.strip()
data['ONE-LINE'] = data['ONE-LINE'].str.replace('\n','').str.strip()
data['STARS'] = data['STARS'].str.replace('\n','').str.strip()

In [ ]:
data[['GENRE','ONE-LINE','STARS']].head()

- Then we will change YEAR column, where we extract only digits and change to int type

In [ ]:
data['YEAR'] = data['YEAR'].str.extract('(\d{4})').astype('str')
data['YEAR'] = pd.to_datetime(data['YEAR'], format='%Y', errors='coerce').dt.year.astype('Int64')

In [ ]:
data['YEAR'].head()

- Now we will take STARS column and separate directors and stars into 2 separate columns

In [ ]:
def extract_info(s):
    info = {'Directors':'', 'Stars':''}

    parts = [p.strip() for p in s.split('|')]
    for part in parts:
        if part.startswith('Director') or part.startswith('Directors'):
            info['Directors'] = part.split(':',1)[1].strip()
        if part.startswith('Star') or part.startswith('Stars'):
            info['Stars'] = part.split(':',1)[1].strip()
    return pd.Series(info)


data[['Directors', 'Stars']] = data['STARS'].apply(extract_info)
data.drop(columns='STARS', inplace=True)

In [ ]:
data[['Directors', 'Stars']] = data[['Directors', 'Stars']].replace("","Unknown")

In [ ]:
data[['Directors','Stars']].head()

- We will look at GROSS column and after checking if any gross exceeded milions, we will strip value of useless signs, convert dollars into euros and change its type to float

In [ ]:
known_gross = data[(data['Gross'].notna()) & (~data['Gross'].str.endswith('M', na=False))]
known_gross

In [ ]:

data.rename(columns={'Gross':'Gross_EUR'}, inplace=True)

def clean_val(val):
    if pd.notna(val):
        float_val = float(str(val).replace('$', '').replace('M', ''))
        currency_change = (float_val * 1000000)*0.85
        return currency_change
    else:
        return val
    
data['Gross_EUR'] = data['Gross_EUR'].apply(clean_val)
data['Gross_EUR']=data['Gross_EUR'].fillna(0)
data['Gross_EUR'] = data['Gross_EUR'].astype(float)


In [ ]:
data['Gross_EUR'].info()

- Remove duplicates

In [ ]:
print(data.duplicated().sum())
data.drop_duplicates(inplace=True)

- We remove ',' from VOTES rows and change it to float

In [ ]:
data['VOTES'] = data['VOTES'].astype('str').str.replace(',','').str.strip()
data['VOTES'] = pd.to_numeric(data['VOTES'], errors='coerce')
data['VOTES'].fillna(0, inplace=True)

- We fill missing GENRE values with the most common genre for each director, and any remaining NaNs are set to 'Unknown'.

In [ ]:
data['Directors'].value_counts()

In [ ]:
mode_genre = data.groupby('Directors')['GENRE'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')
data['GENRE'] = data['GENRE'].fillna(data['Directors'].map(mode_genre))
data['GENRE'] = data['GENRE'].fillna('Unknown')  
data['GENRE'].isna().sum()

- Impute RunTime, RATING and YEAR columns with medians

In [ ]:
print(data['RunTime'].mean())
print(data['RATING'].mean())
print(int(data['YEAR'].mean()))

In [ ]:
data['RunTime'] = data['RunTime'].fillna(round(data['RunTime'].mean(), 1))
data['RATING'] = data['RATING'].fillna(round(data['RATING'].mean(), 1))
data['YEAR'] = data['YEAR'].fillna(int(data['YEAR'].mean()))


- In ONE-LINE columns, we change Add a Plot rows with Unknown

In [ ]:
data['ONE-LINE'] = data['ONE-LINE'].str.replace('Add a Plot', 'Unknown')

In [ ]:
print("Missing values now:\n------------------")
missing_values()

In [ ]:
data.head(10)

In [ ]:
data.info()

- Now we see we eliminated all missing values and hopefully cleaned dataset enough so future ml model will be as precise as it can be

### =============================================
### 4. EXPLORATORY DATA ANALYSIS
### =============================================